## Construct pydantic model from text input

In [2]:
from pydantic_ai import Agent

agent = Agent(model="google-gla:gemini-2.5-flash")

result = await agent.run("Give me and IT employee working in Sweden, keep it short")
result

AgentRunResult(output='**Lars Nilsson**\n\nLars, mid-30s, is a Cloud Engineer at a mid-sized tech company in Stockholm. He specializes in optimizing AWS infrastructure, automating deployments with Terraform, and is never far from his next *fika*.')

In [4]:
print(result.output)

**Lars Nilsson**

Lars, mid-30s, is a Cloud Engineer at a mid-sized tech company in Stockholm. He specializes in optimizing AWS infrastructure, automating deployments with Terraform, and is never far from his next *fika*.


In [5]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int = Field(gt=30_000, lt=50_000)
    position: str

result = await agent.run(
    "Give me an employee working in SWeden", 
    output_type=EmployeeModel
)

result

AgentRunResult(output=EmployeeModel(name='John Doe', age=30, salary=45000, position='Software Engineer'))

In [7]:
employee = result.output
employee

EmployeeModel(name='John Doe', age=30, salary=45000, position='Software Engineer')

In [9]:
employee.name, employee.age, employee.position

('John Doe', 30, 'Software Engineer')

In [10]:
employee.model_dump()

{'name': 'John Doe',
 'age': 30,
 'salary': 45000,
 'position': 'Software Engineer'}

In [11]:
employee.model_dump_json()

'{"name":"John Doe","age":30,"salary":45000,"position":"Software Engineer"}'

## handle several employees OR a list of employees

In [12]:
result = await agent.run(
    """Give me 10 employees in AI and data engineering fields, 
    roles can vary but salary must be between 30000 and 50000""",
    output_type=list[EmployeeModel]
)

employees = result.output
employees

[EmployeeModel(name='Alice Smith', age=30, salary=45000, position='AI Engineer'),
 EmployeeModel(name='Bob Johnson', age=35, salary=48000, position='Data Engineer'),
 EmployeeModel(name='Charlie Brown', age=28, salary=40000, position='Machine Learning Engineer'),
 EmployeeModel(name='Diana Prince', age=32, salary=42000, position='ETL Developer'),
 EmployeeModel(name='Eve Adams', age=38, salary=49000, position='Big Data Engineer'),
 EmployeeModel(name='Frank White', age=29, salary=38000, position='AI Researcher'),
 EmployeeModel(name='Grace Lee', age=33, salary=46000, position='Data Architect'),
 EmployeeModel(name='Harry Green', age=27, salary=39000, position='Deep Learning Engineer'),
 EmployeeModel(name='Ivy Black', age=31, salary=43000, position='Machine Learning Scientist'),
 EmployeeModel(name='Jack Blue', age=36, salary=47000, position='Senior Data Engineer')]

In [13]:
len(employees)

10

In [ ]:
[print(f"{employee.name = } and {employee.salary = }") for employee in employees]

#NOTE: You see [None, None, ...] because print() returns None — 
# the list comprehension collects those return values 
# and Jupyter displays the resulting list. 
# Use a plain for-loop (recommended) or build a list of strings instead.

employee.name = 'Alice Smith' and employee.salary = 45000
employee.name = 'Bob Johnson' and employee.salary = 48000
employee.name = 'Charlie Brown' and employee.salary = 40000
employee.name = 'Diana Prince' and employee.salary = 42000
employee.name = 'Eve Adams' and employee.salary = 49000
employee.name = 'Frank White' and employee.salary = 38000
employee.name = 'Grace Lee' and employee.salary = 46000
employee.name = 'Harry Green' and employee.salary = 39000
employee.name = 'Ivy Black' and employee.salary = 43000
employee.name = 'Jack Blue' and employee.salary = 47000


[None, None, None, None, None, None, None, None, None, None]

In [18]:
for employee in employees:
    print(f"{employee.name = } and {employee.salary = }")

employee.name = 'Alice Smith' and employee.salary = 45000
employee.name = 'Bob Johnson' and employee.salary = 48000
employee.name = 'Charlie Brown' and employee.salary = 40000
employee.name = 'Diana Prince' and employee.salary = 42000
employee.name = 'Eve Adams' and employee.salary = 49000
employee.name = 'Frank White' and employee.salary = 38000
employee.name = 'Grace Lee' and employee.salary = 46000
employee.name = 'Harry Green' and employee.salary = 39000
employee.name = 'Ivy Black' and employee.salary = 43000
employee.name = 'Jack Blue' and employee.salary = 47000


#### CV or resume model - a more complex and nested model

In [21]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int

class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int

class CvModel(BaseModel):
    name: str
    age: int
    experience: list[ExperienceModel]
    educations: list[EducationModel]

result = await agent.run(
    "Create a swedish person applying for a data engineering position",
    output_type=CvModel
)

resume = result.output
resume

CvModel(name='Erik Karlsson', age=32, experience=[ExperienceModel(title='Data Engineer', company='Spotify', description='Developed and maintained data pipelines.', start_year=2018, end_year=2023), ExperienceModel(title='Junior Data Engineer', company='H&M', description='Assisted in data warehouse management.', start_year=2016, end_year=2018)], educations=[EducationModel(title='MSc in Computer Science', education_area='Data Engineering', school='KTH Royal Institute of Technology', description='Specialized in distributed systems and data modeling.', start_year=2013, end_year=2015), EducationModel(title='BSc in Software Development', education_area='Computer Science', school='Uppsala University', description='Focused on algorithms and data structures.', start_year=2010, end_year=2013)])

In [22]:
resume.name, resume.age

('Erik Karlsson', 32)

In [23]:
resume.experience

[ExperienceModel(title='Data Engineer', company='Spotify', description='Developed and maintained data pipelines.', start_year=2018, end_year=2023),
 ExperienceModel(title='Junior Data Engineer', company='H&M', description='Assisted in data warehouse management.', start_year=2016, end_year=2018)]

In [25]:
resume.experience[0].title

'Data Engineer'

## Optional postprocessing: load into duckdb and unnest

In [26]:
resume.model_dump().keys()

dict_keys(['name', 'age', 'experience', 'educations'])

In [28]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="resume_json_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging"
)

info = pipeline.run(
    data=[resume.model_dump()],
    loader_file_format="jsonl",
    table_name="cv_entries"
    )

print(info)

Pipeline resume_json_duckdb load step completed in 0.23 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\Katrin\Documents\github\ai_engineering_katrin_rylander_de24\07_pydanticai_fundamentals\cv.duckdb location to store data
Load package 1764762582.0575218 is LOADED and contains no failed jobs


In [29]:
import duckdb

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc").df()

In [30]:
desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__educations,"[title, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experience,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [31]:
with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc").df()
    cv_entries = conn.sql("from staging.cv_entries").df()
    education = conn.sql("from staging.cv_entries__educations").df()
    experience = conn.sql("from staging.cv_entries__experience").df()


In [32]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,Erik Karlsson,32,1764762582.0575218,pZupZ9XjaZOCAg


In [33]:
education

,title,education_area,school,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,MSc in Computer Science,Data Engineering,KTH Royal Institute of Technology,Specialized in distributed systems and data mo...,2013,2015,pZupZ9XjaZOCAg,0,jNr7EVMXRPNmCg
1,BSc in Software Development,Computer Science,Uppsala University,Focused on algorithms and data structures.,2010,2013,pZupZ9XjaZOCAg,1,APhZcrNQX4tWPw


In [34]:
experience

,title,company,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Data Engineer,Spotify,Developed and maintained data pipelines.,2018,2023,pZupZ9XjaZOCAg,0,WIfxTWpCdsRJUg
1,Junior Data Engineer,H&M,Assisted in data warehouse management.,2016,2018,pZupZ9XjaZOCAg,1,zAXcDDRE/3CFaQ


In [37]:
duckdb.sql("""
    SELECT
        cv.name,
        cv.age,
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        e.title,
        e.education_area,
        e.school,
        e.start_year AS education_start_year,
        e.end_year AS education_end_year
    FROM cv_entries cv
    LEFT JOIN education e ON cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experience ex ON cv._dlt_id = ex._dlt_parent_id      
""").df()

,name,age,company,experience_description,experience_start_year,experience_end_year,title,education_area,school,education_start_year,education_end_year
0,Erik Karlsson,32,H&M,Assisted in data warehouse management.,2016,2018,MSc in Computer Science,Data Engineering,KTH Royal Institute of Technology,2013,2015
1,Erik Karlsson,32,H&M,Assisted in data warehouse management.,2016,2018,BSc in Software Development,Computer Science,Uppsala University,2010,2013
2,Erik Karlsson,32,Spotify,Developed and maintained data pipelines.,2018,2023,MSc in Computer Science,Data Engineering,KTH Royal Institute of Technology,2013,2015
3,Erik Karlsson,32,Spotify,Developed and maintained data pipelines.,2018,2023,BSc in Software Development,Computer Science,Uppsala University,2010,2013
